# Filter Kaggle images for Roboflow

Notebook ini menyiapkan subset UCF Crime yang lebih kecil, seimbang, dan siap diunggah ke Roboflow. Folder hasil memakai nama kelas sehingga cocok untuk proyek **single-label classification**. Untuk object detection, gambar tetap dapat dipakai tetapi bounding box perlu dianotasi di Roboflow.

Dataset Acoustic Extinguisher tidak diproses karena hanya berisi data sensor/tabular. Dataset ArtiFact juga tidak dicampur karena targetnya klasifikasi gambar real vs sintetis, bukan insiden darurat.

In [ ]:
!pip -q install kagglehub pillow tqdm

## Pengaturan
Pilih kelas yang sesuai dengan workflow Roboflow Anda. `MAX_PER_CLASS` membatasi jumlah gambar per kelas, sedangkan `KEEP_EVERY_NTH` mengurangi frame berurutan yang sangat mirip.

In [ ]:
DATASET_HANDLE = "odins0n/ucf-crime-dataset"

SELECTED_CLASSES = [
    "Abuse", "Arson", "Assault", "Burglary", "Explosion",
    "Fighting", "RoadAccidents", "Robbery", "Shooting",
    "Shoplifting", "Stealing", "Vandalism",
]

MAX_PER_CLASS = 1000       # Turunkan untuk uji coba awal
KEEP_EVERY_NTH = 10       # Dataset sudah mengambil tiap frame ke-10; ini menjarangkan lagi
RANDOM_SEED = 42
OUTPUT_NAME = "ucf_crime_filtered_for_roboflow"

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import random
import re
import shutil

import kagglehub
from PIL import Image
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
dataset_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
print(f"Dataset tersedia di: {dataset_root}")

## Temukan kelas dan buat kandidat
Pencocokan nama dibuat toleran terhadap spasi, garis bawah, dan perbedaan huruf besar-kecil.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", value.lower())

wanted = {normalize_name(name): name for name in SELECTED_CLASSES}
grouped = defaultdict(list)
all_images = [p for p in dataset_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]

for image_path in all_images:
    matched_class = None
    for part in reversed(image_path.parts[:-1]):
        key = normalize_name(part)
        if key in wanted:
            matched_class = wanted[key]
            break
    if matched_class:
        grouped[matched_class].append(image_path)

print(f"Total gambar ditemukan: {len(all_images):,}")
for class_name in SELECTED_CLASSES:
    print(f"{class_name:16s}: {len(grouped[class_name]):,}")

missing = [name for name in SELECTED_CLASSES if not grouped[name]]
if missing:
    print("\nKelas tidak ditemukan:", missing)

## Jarangkan frame dan batasi tiap kelas
Urutan file dipertahankan sebelum penjarangan, lalu sampel diacak secara reproduktif.

In [ ]:
selected = []
for class_name in SELECTED_CLASSES:
    candidates = sorted(grouped[class_name])
    candidates = candidates[::max(1, KEEP_EVERY_NTH)]
    if len(candidates) > MAX_PER_CLASS:
        candidates = random.sample(candidates, MAX_PER_CLASS)
    selected.extend((class_name, path) for path in candidates)

print(f"Kandidat terpilih: {len(selected):,}")
print(Counter(class_name for class_name, _ in selected))

## Validasi, hapus duplikat persis, dan salin
File sumber tidak diubah. Hasil disalin ke folder baru dengan manifest CSV.

In [ ]:
output_root = Path("/content") / OUTPUT_NAME
if output_root.exists():
    shutil.rmtree(output_root)
output_root.mkdir(parents=True)

seen_hashes = set()
manifest_rows = []
stats = Counter()

for class_name, source_path in tqdm(selected, desc="Memvalidasi gambar"):
    try:
        with Image.open(source_path) as image:
            image.verify()
        file_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
        if file_hash in seen_hashes:
            stats["duplicate"] += 1
            continue
        seen_hashes.add(file_hash)

        class_dir = output_root / class_name
        class_dir.mkdir(exist_ok=True)
        target_name = f"{file_hash[:10]}_{source_path.name}"
        target_path = class_dir / target_name
        shutil.copy2(source_path, target_path)
        manifest_rows.append({
            "class": class_name,
            "file": str(target_path.relative_to(output_root)),
            "source": str(source_path.relative_to(dataset_root)),
            "sha256": file_hash,
        })
        stats["valid"] += 1
    except Exception:
        stats["invalid"] += 1

with (output_root / "manifest.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["class", "file", "source", "sha256"])
    writer.writeheader()
    writer.writerows(manifest_rows)

print(stats)
print("Hasil per kelas:", Counter(row["class"] for row in manifest_rows))

## Buat ZIP dan unduh
Periksa ringkasan kelas di atas sebelum mengunduh. ZIP dapat diunggah ke Roboflow melalui halaman upload.

In [ ]:
archive_path = shutil.make_archive(str(output_root), "zip", root_dir=output_root)
print(f"ZIP selesai: {archive_path}")

from google.colab import files
files.download(archive_path)

# Dataset privasi: wajah dan plat nomor

Bagian opsional ini membuat dua dataset **object detection** terpisah untuk model anonimisasi:

- `privacy-face`: deteksi wajah sebelum blur.
- `privacy-license-plate`: deteksi plat sebelum blur.

Dataset tidak digabungkan dengan kelas insiden. Jalankan bagian ini terpisah dari UCF Crime agar ruang penyimpanan Colab tidak cepat habis.

In [ ]:
PRIVACY_DATASETS = {
    "privacy-face": {
        "handle": "fareselmenshawii/face-detection-dataset",
        "class_name": "face",
    },
    "privacy-license-plate": {
        "handle": "barkataliarbab/license-plate-detection-dataset-10125-images",
        "class_name": "license_plate",
    },
}

PRIVACY_MAX_IMAGES_PER_DATASET = 5000
PRIVACY_RANDOM_SEED = 42

## Unduh dan buat subset YOLO
Hanya pasangan gambar-label yang lengkap yang disalin. Nama file dibuat unik dan hasil setiap model dimasukkan ke ZIP tersendiri.

In [ ]:
from collections import defaultdict
from pathlib import Path
import hashlib
import random
import shutil

import kagglehub
from PIL import Image
from tqdm.auto import tqdm

PRIVACY_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}
privacy_output_root = Path("/content/privacy_datasets_for_roboflow")
privacy_output_root.mkdir(parents=True, exist_ok=True)
privacy_archives = {}

for project_name, config in PRIVACY_DATASETS.items():
    print(f"\nMenyiapkan {project_name}...")
    source_root = Path(kagglehub.dataset_download(config["handle"]))
    images = [p for p in source_root.rglob("*") if p.suffix.lower() in PRIVACY_IMAGE_EXTENSIONS]
    labels_by_stem = defaultdict(list)
    for label_path in source_root.rglob("*.txt"):
        if label_path.name.lower() not in {"classes.txt", "readme.txt"}:
            labels_by_stem[label_path.stem].append(label_path)

    pairs = [(image, labels_by_stem[image.stem][0]) for image in images if labels_by_stem[image.stem]]
    random.Random(PRIVACY_RANDOM_SEED).shuffle(pairs)
    pairs = pairs[:PRIVACY_MAX_IMAGES_PER_DATASET]

    project_root = privacy_output_root / project_name
    if project_root.exists():
        shutil.rmtree(project_root)
    images_dir = project_root / "images"
    labels_dir = project_root / "labels"
    images_dir.mkdir(parents=True)
    labels_dir.mkdir(parents=True)

    copied = 0
    invalid = 0
    for image_path, label_path in tqdm(pairs, desc=project_name):
        try:
            with Image.open(image_path) as image:
                image.verify()
            label_text = label_path.read_text(encoding="utf-8").strip()
            if not label_text:
                invalid += 1
                continue
            unique_id = hashlib.sha1(str(image_path).encode()).hexdigest()[:12]
            target_stem = f"{unique_id}_{image_path.stem}"
            shutil.copy2(image_path, images_dir / f"{target_stem}{image_path.suffix.lower()}")
            (labels_dir / f"{target_stem}.txt").write_text(label_text + "\n", encoding="utf-8")
            copied += 1
        except Exception:
            invalid += 1

    data_yaml = (
        "path: .\n"
        "train: images\n"
        "nc: 1\n"
        f"names: ['{config['class_name']}']\n"
    )
    (project_root / "data.yaml").write_text(data_yaml, encoding="utf-8")
    archive = shutil.make_archive(str(project_root), "zip", root_dir=project_root)
    privacy_archives[project_name] = archive
    print(f"{copied:,} pasangan valid; {invalid:,} dilewati; ZIP: {archive}")

## Unduh ZIP privasi
Pilih satu nama project lalu jalankan sel. Setelah diunggah ke Roboflow, buat dua model deteksi dan sambungkan prediksinya ke `Blur Visualization` pada Workflow.

In [ ]:
PRIVACY_PROJECT_TO_DOWNLOAD = "privacy-face"  # atau: privacy-license-plate

from google.colab import files
files.download(privacy_archives[PRIVACY_PROJECT_TO_DOWNLOAD])